# **TRAINING NOTEBOOK**

---

In this notebook you can run training on the LIBERO datasets for the ActionJEPA VLA.
Follow ALL the instructions in the notebook!


## **IMPORT**

In [1]:
import os
import random
import json
from utils.utils import preprocess_libero_dataset
from Dataset.LiberoDataset import LiberoDataset
from model.ActionJEPA import ActionJEPA
from training.train import train
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
import numpy as np
import cv2

# REPRODUCIBILITY
seed = 46

# Set seed for torch, numpy and random libraries
torch.manual_seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
np.random.seed(seed)
random.seed(seed)

# Set the devide mode on GPU (if available CUDA for Nvidia and  MPS for Apple Silicon) or CPU
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

datasets_dir = "./LIBERO/libero/datasets" 
processed_data_dir ='./processed_data'
checkpoints_path = "./checkpoints" 


/home/cyberm/miniconda3/envs/myenv/lib/python3.10/site-packages/wandb/sdk/internal/internal_api.py:30: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


## HYPERPARAMETERS CONFIG

In [2]:
# Loading the config.json file
with open('config.json', 'r') as f:
    config = json.load(f)

# Hyperparameters definition
NUM_FRAMES = config['num_frames']
NUM_EPOCHS = config['num_epochs']
BATCH_SIZE = config['batch_size']
LEARNING_RATE = config['learning_rate']
LAMBDA_ACTOR = config['lambda_actor']
LAMBDA_REFINER = config['lambda_refiner']
DATASET_TYPE = config['dataset_type']   # it can be "libero_10", "libero_90", "libero_spatial", "libero_object", "libero_goal" or "all"
INTERPOLATION_TYPE = config['interpolation_type']   # it can be 0,1,2,3 selection the corresponding method in the interpolation_methods_list

interpolation_methods_list = [cv2.INTER_NEAREST, cv2.INTER_LINEAR, cv2.INTER_AREA, cv2.INTER_CUBIC]
interpolation = interpolation_methods_list[INTERPOLATION_TYPE]

if DATASET_TYPE == "all":
  selected_tasks = ["libero_10", "libero_90", "libero_spatial", "libero_goal", "libero_object"]
else:
  selected_tasks = [DATASET_TYPE]
  
print("="*40)
print(f"✅ Training config created!")
print(f"Epochs: {NUM_EPOCHS} | Frames: {NUM_FRAMES} | Batch Size: {BATCH_SIZE}")
print(f"LR: {LEARNING_RATE} | Lambda Actor: {LAMBDA_ACTOR} | Lambda Refiner: {LAMBDA_REFINER}")
print(f"Interpolation method: {str(interpolation)}")
print("="*40)

✅ Training config created!
Epochs: 50 | Frames: 16 | Batch Size: 32
LR: 0.001 | Lambda Actor: 1.0 | Lambda Refiner: 1.0
Interpolation method: 1


## **DATASET PREPROCESSING**



In [ ]:
libero_paths = [f"{datasets_dir}/{task}/*.hdf5" for task in selected_tasks]

if use_backbone:
    # Path for the VJEPA Vision Encoder
    vjepa_path = "checkpoints/facebook/vjepa2-vitg-fpc64-256"

    vision_backbone = VJEPAEncoder(
        model_path=vjepa_path,
        frozen=True,
        device=device
    ).to(device)

    # Path for the CLIP Language Encoder 
    clip_path = "checkpoints/openai/clip-vit-large-patch14"

    language_backbone = CLIPEncoder(
        model_path=clip_path,
        frozen=True,
        device=device
    ).to(device)

    for path in libero_paths:
        preprocess_libero_dataset(hdf5_path=path,
                                output_dir=processed_data_dir,
                                vision_backbone = vision_backbone, 
                                language_backbone = language_backbone, 
                                use_backbone = True
                                )
else:
    for path in libero_paths:
        preprocess_libero_dataset(hdf5_path=path,
                                output_dir=processed_data_dir,
                                )

put_the_bowl_on_top_of_the_cabinet_demo: 100%|██████████| 50/50 [00:00<00:00, 54.67it/s]


## **DATASET CLASS AND SPLIT**

In [5]:
if DATASET_TYPE == "all":
  selected_tasks = ["libero_10", "libero_90", "libero_spatial", "libero_goal", "libero_object"]
else:
  selected_tasks = [DATASET_TYPE]
  
processed_data_dir ='./processed_data'

dataset = LiberoDataset(data_dir=processed_data_dir,
                        selected_tasks=selected_tasks,
                        window_size=NUM_FRAMES,
                        stride=1,
                        use_features = False
                        )

train_percentage = 0.7
val_percentage = 0.2

train_size = int(train_percentage*len(dataset))
val_size = int(val_percentage*len(dataset))
test_size = len(dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(dataset=dataset, lengths=[train_size, val_size, test_size])

print(f"Total dataset size (chunks): {len(dataset)}, Train Size (chunks): {len(train_dataset)}, Validation Size (chunks): {len(val_dataset)}, Test Size (steps): {len(test_dataset)}")

Total dataset size (chunks): 56228, Train Size (chunks): 39359, Validation Size (chunks): 11245, Test Size (steps): 5624


## **MODEL DEFINITION**

In [6]:
# Path for all the models
vjepa_path = os.path.join(checkpoints_path,"facebook/vjepa2-vitg-fpc64-256")
vjepa_pred_path = os.path.join(checkpoints_path,"facebook/jepa-wms/vjepa2_ac_droid.pth.tar/vjepa2_ac_droid.pth.tar")
clip_path = os.path.join(checkpoints_path,"openai/clip-vit-large-patch14")

model = ActionJEPA(
    vjepa_encoder_path=vjepa_path,
    vjepa_predictor_path=vjepa_pred_path,
    clip_model_path=clip_path,
    num_frames=NUM_FRAMES,
    use_backbone = True,
    device=device
).to(device)

model.print_model_info()

Loading weights:   0%|          | 0/843 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: ./checkpoints/openai/clip-vit-large-patch14
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


MODEL INFO:

VISION BACKBONE: VJEPAEncoder
VJEPAEncoder(
  (model): VJEPA2Model(
    (encoder): VJEPA2Encoder(
      (embeddings): VJEPA2Embeddings(
        (patch_embeddings): VJEPA2PatchEmbeddings3D(
          (proj): Conv3d(3, 1408, kernel_size=(2, 16, 16), stride=(2, 16, 16))
        )
      )
      (layer): ModuleList(
        (0-39): 40 x VJEPA2Layer(
          (norm1): LayerNorm((1408,), eps=1e-06, elementwise_affine=True)
          (attention): VJEPA2RopeAttention(
            (query): Linear(in_features=1408, out_features=1408, bias=True)
            (key): Linear(in_features=1408, out_features=1408, bias=True)
            (value): Linear(in_features=1408, out_features=1408, bias=True)
            (proj): Linear(in_features=1408, out_features=1408, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
          (drop_path): Identity()
          (norm2): LayerNorm((1408,), eps=1e-06, elementwise_affine=True)
          (mlp): VJEPA2MLP(
            (fc1): L

## **TRAIN**

In [ ]:
with open('config.json', 'r') as f:
    config = json.load(f)

# Name of the directory for the results
results_dir_path = "./results"
os.makedirs(results_dir_path, exist_ok=True)

# Definition of the Categorical Cross Entropy Loss
loss_fn = nn.MSELoss()
# Definition of the optimizer
optimizer = torch.optim.Adam(model.parameters(), lr = LEARNING_RATE)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=16,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=16,
    pin_memory=True
)

train(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    loss_fn=loss_fn,
    num_epochs=NUM_EPOCHS,
    config=config,
    device=device,
    results_dir_path=results_dir_path,
    lambda_actor=LAMBDA_ACTOR,
    lambda_refiner=LAMBDA_REFINER

)


EPOCH: 1/50


Training:   0%|          | 0/1230 [00:00<?, ?it/s]